In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
from os.path import join

import pandas as pd
import numpy as np

import missingno as msno

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score
import xgboost as xgb
import lightgbm as lgb

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train_data_path = join('../input', 'train.csv')
sub_data_path = join('../input', 'test.csv')

## 1. ??? ????
pandas? read_csv ??? ??? ???? ????, ? ???? ???? ??? ???????.
1. ID : ?? ???? ??
2. date : ?? ??? ??
3. price : ?? ??? ?? ??
4. bedrooms : ??? ?
5. bathrooms : ??? ??? ??
6. sqft_living : ?? ??? ?? ??
7. sqft_lot : ??? ?? ??
8. floors : ?? ? ?
9. waterfront : ?? ??? ?? ???? ?? (a.k.a. ???)
10. view : ?? ??? ?? ????? ??
11. condition : ?? ???? ??
12. grade : King County grading ??? ???? ?? ?? ??
13. sqft_above : ???? ??? ?? ??
14. sqft_basement : ???? ?? ??
15. yr_built : ?? ?? ??
16. yr_renovated : ?? ???? ??
17. zipcode : ????
18. lat : ??
19. long : ??
20. sqft_living15 : 2015? ?? ?? ??? ?? ??(?? ??????, ??? ?? ? ??)
21. sqft_lot15 : 2015? ?? ??? ?? ??(?? ??????, ??? ?? ? ??)

In [ ]:
data = pd.read_csv(train_data_path)
sub = pd.read_csv(sub_data_path)
print('train data dim : {}'.format(data.shape))
print('sub data dim : {}'.format(sub.shape))

In [ ]:
y = data['price']

del data['price']

In [ ]:
train_len = len(data)
data = pd.concat((data, sub), axis=0)

In [ ]:
data.head()

## 2. ??? ??? 
? ???? ?? ?? ??? ????, ??? ?????? ???? ???? ?????.
### ??? ??
?? ???? ???? ???? ???????.<br>
missingno ?????? matrix ??? ????, ???? ?? ??? ???? ?? ??? ? ????.

In [ ]:
msno.matrix(data)

?? ??? ???? ?? ??? ????, ?? ??? ???? ???????.<br>

In [ ]:
for c in data.columns:
    print('{} : {}'.format(c, len(data.loc[pd.isnull(data[c]), c].values)))

### id, date ?? ??
id ??? ??? ??? ????? ??? ?? ???? ?????.<br>
date ??? ??????? ?? ??? ???, ??? ???? ??? ??? ??????.

In [ ]:
sub_id = data['id'][train_len:]
del data['id']
data['date'] = data['date'].apply(lambda x : str(x[:6])).astype(str)

### ? ???? ?? ??
???? ??? ??? ??? ??? ????? ?? ?? ??? ???? ???? ??? ????.

In [ ]:
fig, ax = plt.subplots(10, 2, figsize=(20, 60))

# id ??? ???? ??? ?????.
count = 0
columns = data.columns
for row in range(10):
    for col in range(2):
        sns.kdeplot(data[columns[count]], ax=ax[row][col])
        ax[row][col].set_title(columns[count], fontsize=15)
        count+=1
        if count == 19 :
            break

price, bedrooms, sqft_living, sqft_lot, sqft_above, sqft_basement ??? ???? ??? ??? ?????.<br>
log-scaling? ?? ??? ??? ????? ??? ??? ?????.

In [ ]:
skew_columns = ['bedrooms', 'sqft_living', 'sqft_lot', 'sqft_above', 'sqft_basement']

for c in skew_columns:
    data[c] = np.log1p(data[c].values)

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(10, 15))

count = 0
for row in range(3):
    for col in range(2):
        if count == 5:
            break
        sns.kdeplot(data[skew_columns[count]], ax=ax[row][col])
        ax[row][col].set_title(skew_columns[count], fontsize=15)
        count+=1



???? ???? ??? ??? ??? ? ????.

In [ ]:
sub = data.iloc[train_len:, :]
x = data.iloc[:train_len, :]

## 3. ???
### Average Blending
???? ??? ??? ?? ??? ?? Blending ??? ??????.

In [ ]:
gboost = GradientBoostingRegressor(random_state=2019)
xgboost = xgb.XGBRegressor(random_state=2019)
lightgbm = lgb.LGBMRegressor(random_state=2019)

models = [{'model':gboost, 'name':'GradientBoosting'}, {'model':xgboost, 'name':'XGBoost'},
          {'model':lightgbm, 'name':'LightGBM'}]

### Cross Validation
?? ??? ?? ??? ??? ??? ???????.

In [ ]:
def get_cv_score(models):
    kfold = KFold(n_splits=5, random_state=2019).get_n_splits(x.values)
    for m in models:
        print("Model {} CV score : {:.4f}".format(m['name'], np.mean(cross_val_score(m['model'], x.values, y)), 
                                             kf=kfold))

In [ ]:
get_cv_score(models)

### Make Submission

?? ??? ???? cross_val_score ??? R<sup>2</sup>? ?????.<br>
R<sup>2</sup> ?? 1? ????? ??? ???? ? ???? ?????. 3? ?? ??? ??? ?? ???? ?? ??? ??? ???? ????.<br> ?? ?????? 3? ??? ?????, Average Blending? ?? ?? ??? ??????.

In [ ]:
def AveragingBlending(models, x, y, sub_x):
    for m in models : 
        m['model'].fit(x.values, y)
    
    predictions = np.column_stack([
        m['model'].predict(sub_x.values) for m in models
    ])
    return np.mean(predictions, axis=1)

In [ ]:
y_pred = AveragingBlending(models, x, y, sub)

In [ ]:
sub = pd.DataFrame(data={'id':sub_id,'price':y_pred})

In [ ]:
sub.to_csv('submission.csv', index=False)